# Where does GPU memory go when you train an LLM?

A hands-on experiment: train the same small language model with seven different memory-saving techniques, measure exactly where the memory goes, and see which techniques actually help.

**What you'll measure**
1. **Anatomy of one training step:** how much memory the weights, activations, gradients and optimizer states each take
2. **Prediction vs. reality:** calculate memory needs first, then compare with measurements
3. **Seven training setups:** full fine-tuning in FP32, mixed precision, gradient checkpointing, gradient accumulation, 8-bit Adam, LoRA, and QLoRA

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell from top to bottom with **Shift + Enter**.

## Step 0: Install libraries

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets
!nvidia-smi

**Important:** when this install cell finishes, click **Runtime → Restart session**, then continue with the next cell. Colab needs the restart to use the newly installed libraries; without it, 8-bit and 4-bit loading can fail with a "requires bitsandbytes" error.

## Step 1: Gradients in ten lines

Before the big model, here's what a gradient actually is.

A model makes a prediction, we measure how wrong it is (the **loss**), and the **gradient** tells us, for each weight, which direction makes the loss smaller and how strongly. The optimizer then nudges each weight a little in that direction. Repeat millions of times, and the model learns.

Here's the whole idea with a single weight `w`. We want `w * x` to equal `y`, so the perfect value is `w = 3`.

In [ ]:
import torch

x, y = torch.tensor(2.0), torch.tensor(6.0)     # we want w * 2 = 6, so w should be 3
w = torch.tensor(0.5, requires_grad=True)       # start with a bad guess
lr = 0.05                                       # learning rate: how big each step is

for step in range(8):
    loss = (w * x - y) ** 2                     # forward pass: how wrong are we?
    loss.backward()                             # backward pass: compute the gradient d(loss)/d(w)
    print(f"step {step}: w = {w.item():.3f}, loss = {loss.item():.3f}, gradient = {w.grad.item():.3f}")
    with torch.no_grad():
        w -= lr * w.grad                        # optimizer step: move w against the gradient
    w.grad = None                               # clear the gradient for the next step

Watch `w` move toward 3 while the loss shrinks. A large language model does exactly this, but for hundreds of millions of weights at once. **Every weight needs its own gradient, and Adam also keeps two running averages per weight.** That's why training needs so much more memory than inference.

## Step 2: Settings and helper functions

We use **Qwen2.5 0.5B**, a small open model (about 500 million parameters) that downloads without a login. It's small enough that even full FP32 fine-tuning fits on a free T4, so we can compare every technique fairly.

All experiments train on **exactly the same text, in the same order**, with the same effective batch size, so differences come only from the technique.

In [ ]:
import importlib.metadata as _md
try:
    print("bitsandbytes", _md.version("bitsandbytes"))
except _md.PackageNotFoundError:
    raise SystemExit("bitsandbytes is missing: run the install cell, then Runtime -> Restart session.")
import gc, time, math, torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B"
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
SEQ_LEN = 256          # tokens per training example
EFFECTIVE_BATCH = 4    # examples per optimizer step, same for every experiment
STEPS = 8              # optimizer steps per experiment
GB = 1024 ** 3

assert torch.cuda.is_available(), "No GPU found. Set Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / GB))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Training text: a small slice of Wikipedia (WikiText-2). Falls back to random tokens if the download fails.
# Memory use doesn't depend on the words, only on the shapes, so either works for the measurements.
n_needed = STEPS * EFFECTIVE_BATCH
try:
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    text = " ".join(t for t in ds["text"][:3000] if t.strip())
    ids = tokenizer(text, return_tensors="pt")["input_ids"][0]
    blocks = ids[: (len(ids) // SEQ_LEN) * SEQ_LEN].view(-1, SEQ_LEN)
    assert len(blocks) >= n_needed
    DATA = blocks[:n_needed]
    print("Using WikiText-2:", DATA.shape)
except Exception as e:
    print("Dataset download failed, using random tokens instead:", e)
    torch.manual_seed(0)
    DATA = torch.randint(0, tokenizer.vocab_size, (n_needed, SEQ_LEN))

def get_batch(step, micro, micro_bs):
    start = step * EFFECTIVE_BATCH + micro * micro_bs
    return DATA[start:start + micro_bs].to("cuda")

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

## Step 3: Anatomy of one training step

Here we do a single FP32 training step by hand and measure GPU memory after each phase:

1. **Load the model:** the weights
2. **Forward pass:** the model saves intermediate results (**activations**) because the backward pass needs them
3. **Backward pass:** a **gradient** is computed for every weight, and activations are freed
4. **Optimizer step:** Adam creates two extra values per weight (the **optimizer states**)

This shows, in one chart, where training memory goes.

In [ ]:
free_memory()
torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to("cuda")
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

snap = {}
snap["1. Weights loaded"] = torch.cuda.memory_allocated() / GB
batch = get_batch(0, 0, EFFECTIVE_BATCH)
out = model(input_ids=batch, labels=batch)
snap["2. After forward (activations)"] = torch.cuda.memory_allocated() / GB
out.loss.backward()
del out
snap["3. After backward (gradients)"] = torch.cuda.memory_allocated() / GB
optimizer.step()
snap["4. After optimizer step (Adam states)"] = torch.cuda.memory_allocated() / GB
peak_anatomy = torch.cuda.max_memory_allocated() / GB

total_params, _ = count_params(model)
print(f"Parameters: {total_params/1e6:.0f} million\n")
for k, v in snap.items():
    print(f"{k:40s} {v:6.2f} GB")
print(f"{'Peak during the step':40s} {peak_anatomy:6.2f} GB")

del model, optimizer, batch
free_memory()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(list(snap.keys()), list(snap.values()), color="#1d5d86")
plt.axhline(peak_anatomy, color="#a95a0c", linestyle="--", label=f"Peak: {peak_anatomy:.2f} GB")
plt.ylabel("GPU memory allocated (GB)")
plt.title("Memory after each phase of one FP32 training step")
plt.xticks(rotation=15, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig("anatomy_training_step.png", dpi=150)
plt.show()

**What to look for:** the weights are only the start. Gradients roughly double the memory, and Adam's states add about twice the weights again. The peak usually happens during the backward pass, when activations and gradients exist at the same time.

## Step 4: Predict before you measure

For full training with the Adam optimizer, the fixed memory per parameter is roughly:

| Part | FP32 bytes per parameter | With 8-bit Adam |
|---|---|---|
| Weights | 4 | 4 |
| Gradients | 4 | 4 |
| Adam states (two values) | 8 | about 2 |
| **Total** | **16** | **about 10** |

On top of that come **activations**, which depend on batch size and sequence length. LoRA and QLoRA change the picture completely, because only a tiny number of parameters is trained.

In [ ]:
P = total_params
predictions = {
    "Full FP32 + AdamW": 16 * P / GB,
    "Full FP32 + 8-bit Adam": 10 * P / GB,
    "LoRA (FP32 base, frozen)": 4 * P / GB,
    "QLoRA (4-bit base, frozen)": 0.5 * P / GB,
}
for k, v in predictions.items():
    print(f"{k:30s} about {v:5.2f} GB before activations")

## Step 5: Seven training setups

One function runs every experiment the same way and records:
* **peak_gb:** the highest GPU memory used
* **sec_per_step:** average time per optimizer step (the first, warm-up step is excluded)
* **final_loss:** loss after the last step, to check the technique didn't break learning
* **trainable_params:** how many weights are actually updated

The techniques:
* **Mixed precision:** compute in 16-bit where safe, keep master weights in 32-bit
* **Gradient checkpointing:** don't store all activations; recompute them during the backward pass
* **Gradient accumulation:** process the batch in smaller pieces and add up the gradients before updating
* **8-bit Adam:** store the optimizer states in 8 bits instead of 32
* **LoRA:** freeze the model and train small adapter matrices
* **QLoRA:** LoRA on top of a 4-bit compressed base model

In [ ]:
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def load_full():
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to("cuda")

def lora_config():
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=LORA_TARGETS, task_type="CAUSAL_LM")

def load_lora():
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to("cuda")
    return get_peft_model(model, lora_config())

def load_qlora():
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, device_map="cuda")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    return get_peft_model(model, lora_config())

def run_experiment(name, loader, amp=False, checkpointing=False, micro_bs=EFFECTIVE_BATCH,
                   optim="adamw", lr=1e-5):
    free_memory()
    torch.manual_seed(0)
    accum = EFFECTIVE_BATCH // micro_bs
    row = {"setup": name}
    model = optimizer = None
    try:
        model = loader()
        if checkpointing:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            model.config.use_cache = False
            if hasattr(model, "enable_input_require_grads"):
                model.enable_input_require_grads()
        for p in model.parameters():          # trainable weights in FP32, so mixed precision is safe
            if p.requires_grad and p.dtype != torch.float32:
                p.data = p.data.float()
        total, trainable = count_params(model)
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = bnb.optim.AdamW8bit(params, lr=lr) if optim == "adamw8bit" else torch.optim.AdamW(params, lr=lr)
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
        model.train()
        torch.cuda.reset_peak_memory_stats()
        step_times, losses = [], []
        for step in range(STEPS):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            step_loss = 0.0
            for micro in range(accum):
                batch = get_batch(step, micro, micro_bs)
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    loss = model(input_ids=batch, labels=batch).loss / accum
                scaler.scale(loss).backward()
                step_loss += loss.item()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize(); step_times.append(time.perf_counter() - t0)
            losses.append(step_loss)
        row.update({
            "peak_gb": round(torch.cuda.max_memory_allocated() / GB, 2),
            "sec_per_step": round(sum(step_times[1:]) / max(1, len(step_times) - 1), 2),
            "first_loss": round(losses[0], 3),
            "final_loss": round(losses[-1], 3),
            "trainable_params": f"{trainable/1e6:.1f}M of {total/1e6:.0f}M",
        })
        row["_losses"] = losses
        print(f"Done: {name}  peak {row['peak_gb']} GB")
    except torch.cuda.OutOfMemoryError:
        row.update({"peak_gb": None, "sec_per_step": None, "first_loss": None, "final_loss": None,
                    "trainable_params": "out of memory"})
        print(f"Out of memory: {name}")
    finally:
        del model, optimizer
        free_memory()
    return row

This runs all seven setups. It takes roughly 10 to 15 minutes.

In [ ]:
experiments = [
    dict(name="1. Full FP32 (baseline)",        loader=load_full),
    dict(name="2. + Mixed precision",           loader=load_full, amp=True),
    dict(name="3. + Gradient checkpointing",    loader=load_full, checkpointing=True),
    dict(name="4. + Gradient accumulation (4x1)", loader=load_full, micro_bs=1),
    dict(name="5. + 8-bit Adam",                loader=load_full, optim="adamw8bit"),
    dict(name="6. LoRA",                        loader=load_lora, amp=True, lr=2e-4),
    dict(name="7. QLoRA",                       loader=load_qlora, amp=True, lr=2e-4),
]
results = [run_experiment(**e) for e in experiments]
df = pd.DataFrame([{k: v for k, v in r.items() if k != "_losses"} for r in results])
df

## Step 6: Compare the results

In [ ]:
plot_df = df.dropna(subset=["peak_gb"])
plt.figure(figsize=(9, 4.5))
bars = plt.barh(plot_df["setup"], plot_df["peak_gb"], color="#1d5d86")
plt.gca().invert_yaxis()
plt.xlabel("Peak GPU memory during training (GB)")
plt.title("Same model, same data: peak memory by technique")
for b, v in zip(bars, plot_df["peak_gb"]):
    plt.text(v + 0.1, b.get_y() + b.get_height() / 2, f"{v:.2f}", va="center")
plt.tight_layout()
plt.savefig("peak_memory_by_technique.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for _, r in plot_df.iterrows():
    plt.scatter(r["peak_gb"], r["sec_per_step"], s=80, color="#1d5d86")
    plt.annotate(r["setup"], (r["peak_gb"], r["sec_per_step"]), textcoords="offset points", xytext=(6, 4), fontsize=8)
plt.xlabel("Peak GPU memory (GB)")
plt.ylabel("Seconds per optimizer step")
plt.title("The trade-off: memory vs. speed")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("memory_vs_speed.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
for r in results:
    if r.get("_losses"):
        plt.plot(range(1, len(r["_losses"]) + 1), r["_losses"], marker="o", label=r["setup"])
plt.xlabel("Optimizer step")
plt.ylabel("Training loss")
plt.title("Did the techniques still learn?")
plt.legend(fontsize=7)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()

**What to look for**
* **Gradient checkpointing** usually saves a lot of memory but makes each step slower, because activations are recomputed.
* **Gradient accumulation** lowers peak memory with the same effective batch, at the cost of more passes per step.
* **8-bit Adam** shrinks the optimizer states, which are a big part of full training memory.
* **Mixed precision** may save less than you expect, or even use more, because the FP32 master weights stay in memory and 16-bit copies are created during computation. Its main benefit on modern GPUs is speed. That's a real finding worth writing about.
* **LoRA and QLoRA** change the game: almost no gradients or optimizer states, because only a tiny fraction of weights is trained. Their loss curves start from a different point because they use a higher learning rate, which is normal.
* Compare these numbers with your **predictions in Step 4**. The difference is mostly activations and temporary buffers.

## Step 7: Save your results

In [ ]:
df.to_csv("results_training_memory.csv", index=False)
pd.DataFrame([snap]).T.rename(columns={0: "GB"}).to_csv("results_anatomy.csv")
print(df.to_markdown(index=False))
print()
print("Saved: results_training_memory.csv, results_anatomy.csv,")
print("       anatomy_training_step.png, peak_memory_by_technique.png, memory_vs_speed.png, loss_curves.png")

## Write down what you found

Answer these in your own words, with your real numbers. They become your README, blog post, and interview answers.

1. In the anatomy chart, which part took the most memory: weights, gradients, optimizer states, or activations?
2. How close were your predictions to the measured peaks? What explains the difference?
3. Which single technique saved the most memory in full fine-tuning? What did it cost in speed?
4. Did mixed precision save memory on this GPU? Why or why not?
5. How much less memory did QLoRA need than the FP32 baseline, and how many parameters did it train?
6. If you had to fine-tune a model three times bigger on the same free GPU, which combination would you choose, and why?